# 01 Temporal feature selection: Two-stage, Lasso trend + random forest

Part of `03b_training_pipeline_model/two_stage_rf`. Run order: **01 feature selection -> 02 tune -> 03 retrain**. Each notebook reads only the config written
by the one before, so any of them can be rerun alone.

The temporal features are the stock aggregates, the economy features, the COE (car licence) features and the annual MOP supply and population counts (`stock_features.csv`, `sg_macro_features_monthly.csv`, `coe_features_monthly.csv`, `annual_mop_population.csv`; 231 + 6 + 12 + 5 candidates).

1. **Decompose the trend.** A Ridge of log rent on one dummy per month plus *all* the static features (`00_static_features/01_encoding`) gives `alpha_m`:
   the monthly level of the average flat with the static differences between flats removed.
2. **Select** the temporal features that explain `alpha_m`: near-duplicates are dropped, then L2 stability selection (a `RidgeCV` on expanding windows of
   months; a feature scores when it is in the top 10 by absolute standardised coefficient; keep those scoring at least 0.5). The top-10 and the 0.5
   threshold were fixed before looking at any results.

Selection is run for **three training windows** (full history from 2021-01, 2023-01 onward, and the months after May 2023 only, i.e. 2023-06 onward) and
**two sets of months**: the train split only (2021-01 to 2024-09, used while tuning against the validation set) and train + validation (to 2025-03, used
for the final model). The six lists are written to this folder's `feature_config.py`. The selection does not depend on the model type, so every model
folder arrives at the same lists; it is repeated here so that each folder runs on its own.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

SHARED_DIR = '/Users/keltonn/Desktop/Projects/housing_price_regressor/03b_training_pipeline_model'
FOLDER = f'{SHARED_DIR}/two_stage_rf'
sys.path.insert(0, SHARED_DIR)      # pipeline_utils.py, shared by every model folder
sys.path.insert(0, FOLDER)          # this folder's feature_config.py and model_config.py

import pipeline_utils as pu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

TOP_K = 10          # fixed in advance
THRESHOLD = 0.5
MIN_MONTHS = 12     # shortest expanding window

## 1. Data

In [2]:
frames, numeric, onehot = pu.load_static()
temporal = pu.load_temporal()
trainval = pd.concat([frames['train'], frames['validation']], ignore_index=True)
data = {
    'train': (pu.design(frames['train'], numeric, onehot), pu.log_rent(frames['train']), frames['train']['month'].to_numpy()),
    'train_val': (pu.design(trainval, numeric, onehot), pu.log_rent(trainval), trainval['month'].to_numpy()),
}
print(f"static design {data['train'][0].shape[1]} columns; train {len(frames['train']):,} rows (months 0..44), "
      f"validation {len(frames['validation']):,} rows (45..50); temporal candidates {temporal.shape[1]} over {len(temporal)} months")

static design 237 columns; train 132,398 rows (months 0..44), validation 16,961 rows (45..50); temporal candidates 254 over 68 months


## 2. Decompose the trend and select

Candidates must be complete over the months used for fitting **and** over every later month up to the end of the dataset (a data-availability filter that
never looks at rent), so every selected feature can be forecast for the test period. 2021-01 is skipped for the trend because `mom_return` needs a
previous month.

In [3]:
def select(window, fit_on):
    X, y, month = data[fit_on]
    last = month.max()
    months = np.arange(pu.WINDOW_START[window], last + 1)
    alpha_m = pu.decompose_trend(X, y, month, months)                  # trend decomposed from the static features
    s1_months = [m for m in months if m >= 1]
    pool = temporal.loc[s1_months + list(range(last + 1, pu.TEST_LAST + 1))]
    pool = pool.loc[:, pool.notna().all()]
    stability = pu.trend_stability(pool, alpha_m.loc[s1_months], top_ks=[TOP_K], min_months=MIN_MONTHS)[TOP_K]
    return alpha_m, pool.shape[1], stability, pu.pick(stability, THRESHOLD)


selected, rows = {}, []
for window in pu.WINDOW_START:
    for fit_on in ('train', 'train_val'):
        alpha_m, n_pool, stability, chosen = select(window, fit_on)
        key = f'{window}/{fit_on}'
        selected[key] = [(name, float(stability[name])) for name in chosen]
        rows.append({'selection': key, 'months': f'{alpha_m.index.min()}..{alpha_m.index.max()}', 'candidates': n_pool,
                     'after de-duplication': len(stability), 'selected': len(chosen),
                     'alpha_m first -> last': f'{alpha_m.iloc[0]:.3f} -> {alpha_m.iloc[-1]:.3f}'})
pd.DataFrame(rows).set_index('selection')

,months,candidates,after de-duplication,selected,alpha_m first -> last
selection,,,,,
full_history/train,0..44,182,175,8,7.608 -> 8.020
full_history/train_val,0..50,182,172,9,7.608 -> 8.036
from_2023/train,24..44,209,191,10,7.940 -> 8.019
from_2023/train_val,24..50,209,192,10,7.940 -> 8.035
from_2023_06/train,29..44,215,195,8,7.987 -> 8.018
from_2023_06/train_val,29..50,215,196,10,7.986 -> 8.034


In [4]:
table = pd.DataFrame({key: pd.Series(dict(feats)) for key, feats in selected.items()})
table.insert(0, 'kind', ['economy' if name.startswith('MACRO_') else 'COE' if name.startswith('coe_') else 'annual' if name.startswith('ANNUAL_') else 'stock' for name in table.index])
table.round(2).fillna('')

,kind,full_history/train,full_history/train_val,from_2023/train,from_2023/train_val,from_2023_06/train,from_2023_06/train_val
MACRO_PRIVATE_RENT_RPI_YOY,economy,0.73,0.62,,,,
MACRO_SORA_3M_PCT,economy,0.85,0.87,1.0,0.75,,
coe_c_log_quota_premium,COE,0.79,0.82,,,,
stk_cmon_return_volatility,stock,,,,,0.6,0.82
stk_comfortdelgro_mom_return,stock,,,1.0,1.0,,
stk_cytomed_therapeutics_mom_return,stock,,,,,,0.55
stk_cytomed_therapeutics_return_volatility,stock,,,,,1.0,1.0
stk_dbs_group_return_volatility,stock,,,,,,0.55
stk_euda_health_log_price_mean,stock,,,0.6,,,
stk_genius_group_return_volatility,stock,,,1.0,1.0,,


## 3. Write `feature_config.py`

In [5]:
lines = [
    '"""Temporal features used by this model folder. Written by 01_feature_selection.ipynb; rerun that notebook to update it."""',
    '',
    '# Stock and economy features selected by L2 stability selection against the trend decomposed from the static features.',
    '# Key = "<training window>/<months the selection used>": "train" = train split only (used while tuning), "train_val" = train + validation',
    f'# (used for the final model). The comment is the share of expanding windows in which the feature was among the top {TOP_K}.',
    'SELECTED_TEMPORAL_FEATURES = {',
]
for key, feats in selected.items():
    lines.append(f"    '{key}': [")
    lines += [f"        '{name}',  # {score:.2f}" for name, score in feats]
    lines.append('    ],')
lines += ['}', '']
with open(f'{FOLDER}/feature_config.py', 'w') as f:
    f.write('\n'.join(lines))
print(open(f'{FOLDER}/feature_config.py').read())

"""Temporal features used by this model folder. Written by 01_feature_selection.ipynb; rerun that notebook to update it."""

# Stock and economy features selected by L2 stability selection against the trend decomposed from the static features.
# Key = "<training window>/<months the selection used>": "train" = train split only (used while tuning), "train_val" = train + validation
# (used for the final model). The comment is the share of expanding windows in which the feature was among the top 10.
SELECTED_TEMPORAL_FEATURES = {
    'full_history/train': [
        'stk_singapore_post_log_price_mean',  # 0.67
        'stk_sembcorp_log_price_mean',  # 0.88
        'MACRO_SORA_3M_PCT',  # 0.85
        'coe_c_log_quota_premium',  # 0.79
        'stk_sheng_siong_group_log_price_mean',  # 0.82
        'stk_keppel_reit_log_price_mean',  # 0.61
        'stk_jardine_cycle_carriage_log_price_mean',  # 0.64
        'MACRO_PRIVATE_RENT_RPI_YOY',  # 0.73
    ],
    'full_history/train_val': [
        